<a href="https://colab.research.google.com/github/hzhoujoy/ST554_HW8/blob/main/JoyZhou_HW8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Home Work 8**   
Author: Huiping Zhou  
Date: 4/3/2026  

# Homework 7

## Instruction
The purpose of this homework is to practice fitting MLR and logistic regression models (including penalized or regularized models).
We will use the `Wine Quality` dataset from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/186/wine+quality). The variables in the dataset based on physicochemical tests include:
- fixed acidity: g(tartaric acid)/dm3
- volatile acidity: g(acetic acid)/dm3
- citric acid: g/dm3
- residual sugar: g/dm3
- chlorides: g(sodium chloride)/dm3
- free sulfur dioxide: mg/dm3
- total sulfur dioxide: mg/dm3
- density: g/cm3
- pH
- sulphates: g(potassium sulphate)/dm3
- alcohol: (vol.%)
- quality: score between 0 to 10
Detialed information can be found in Cortez et al. (2009).

we will use `alcohol` as a target variable for fitting multiple linear regression models while using `type of wine` as the response variable for fitting fitting logistic regression models.

## Read in and Combine Data  
We start by downloading the `winequality-red.csv` and `winequality-white.csv` datasets from the [UCI machine
learning repository site](https://archive.ics.uci.edu/dataset/186/wine+quality), saving them locally, and then uploaingd them into the notebook.

Next, we use the `pd.read_cvs()` function with relative file paths to read in the two datasets.


In [92]:
import pandas as pd
import numpy as np
# read in winequality-red dataset
red_data = pd.read_csv("winequality-red.csv", sep=";")
#Display summary information
red_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1599 entries, 0 to 1598
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1599 non-null   float64
 1   volatile acidity      1599 non-null   float64
 2   citric acid           1599 non-null   float64
 3   residual sugar        1599 non-null   float64
 4   chlorides             1599 non-null   float64
 5   free sulfur dioxide   1599 non-null   float64
 6   total sulfur dioxide  1599 non-null   float64
 7   density               1599 non-null   float64
 8   pH                    1599 non-null   float64
 9   sulphates             1599 non-null   float64
 10  alcohol               1599 non-null   float64
 11  quality               1599 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 150.0 KB


According to the info() output, `red_data` includes 1,599 rows and 12 variables.

In [93]:
#check the first 5 rows of dataset
red_data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5


In [94]:
#read in dataset
white_data = pd.read_csv("winequality-white.csv", sep=";")

#provides a summary of the dataset
white_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4898 entries, 0 to 4897
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         4898 non-null   float64
 1   volatile acidity      4898 non-null   float64
 2   citric acid           4898 non-null   float64
 3   residual sugar        4898 non-null   float64
 4   chlorides             4898 non-null   float64
 5   free sulfur dioxide   4898 non-null   float64
 6   total sulfur dioxide  4898 non-null   float64
 7   density               4898 non-null   float64
 8   pH                    4898 non-null   float64
 9   sulphates             4898 non-null   float64
 10  alcohol               4898 non-null   float64
 11  quality               4898 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 459.3 KB


According to the info() output, `white_data` includes 4,898 rows and 12 columns.

In [95]:
#check the first 5 rows of dataset
white_data.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.0010,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.9940,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.9951,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.9956,3.19,0.40,9.9,6


Then, we combine these two datasets and create a new variable that represents the type of wine (red or white).

In [96]:
# Combine along columns
wine_df = pd.concat([red_data, white_data], ignore_index=True)
wine_df.info() #check summary information of combined dataset

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         6497 non-null   float64
 1   volatile acidity      6497 non-null   float64
 2   citric acid           6497 non-null   float64
 3   residual sugar        6497 non-null   float64
 4   chlorides             6497 non-null   float64
 5   free sulfur dioxide   6497 non-null   float64
 6   total sulfur dioxide  6497 non-null   float64
 7   density               6497 non-null   float64
 8   pH                    6497 non-null   float64
 9   sulphates             6497 non-null   float64
 10  alcohol               6497 non-null   float64
 11  quality               6497 non-null   int64  
dtypes: float64(11), int64(1)
memory usage: 609.2 KB


According to the info() output, combined `wine_df` includes 6497 rows and 12 columns, indicating that the red and white wine datasets were merged properly.

We create a new variable, `type_of_wine`, to indicate whether each observation corresponds to red or white wine type in the combined dataset.

In [97]:
#First part of the combined dataset (rows originally from red_data): label as 0(red)
wine_df.loc[:len(red_data)-1, "type_of_wine"] = 0  #red

# Remaining rows (originally from white_data): label as 1(white)
wine_df.loc[len(red_data):, "type_of_wine"] = 1

In [98]:
wine_df.info()  #check the information of combined dataset

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         6497 non-null   float64
 1   volatile acidity      6497 non-null   float64
 2   citric acid           6497 non-null   float64
 3   residual sugar        6497 non-null   float64
 4   chlorides             6497 non-null   float64
 5   free sulfur dioxide   6497 non-null   float64
 6   total sulfur dioxide  6497 non-null   float64
 7   density               6497 non-null   float64
 8   pH                    6497 non-null   float64
 9   sulphates             6497 non-null   float64
 10  alcohol               6497 non-null   float64
 11  quality               6497 non-null   int64  
 12  type_of_wine          6497 non-null   float64
dtypes: float64(12), int64(1)
memory usage: 660.0 KB


We check if the new variable is created correctly by using 'value_counts() method.

In [99]:
wine_df.type_of_wine.value_counts()

,count
type_of_wine,
1.0,4898
0.0,1599


The `type_of_wine` variable was successfully created as a binary numeric indicator. The value_counts() output confirms that it contains two values (0 and 1), corresponding to red and white wines. The number of observations in each type of wine also sums to the total number of rows in the original dataset, indicating the variable was created correctly.

We update the column names in the `wine_df` DataFrame by replacing spaces with underscores.

In [100]:
wine_df.columns = [col.replace(" ", "_") for col in wine_df.columns]
wine_df.head()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality,type_of_wine
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0.0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,0.0
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,0.0
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,0.0
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0.0


In [101]:
#check if there are missing values in wine_df
wine_df.isnull().sum()

,0
fixed_acidity,0
volatile_acidity,0
citric_acid,0
residual_sugar,0
chlorides,0
free_sulfur_dioxide,0
total_sulfur_dioxide,0
density,0
pH,0
sulphates,0


Based on the missing-value check, the results show that there are no missing values in any column of the dataset.

## Split the data
The `train_test_split()` function from `sklearn.model_selection` is used to split the data into training and test sets.  For the X parts we use the `.drop()` method to select all columns except for `alcohol` which we later use as inputs for the Y parts.  We set `test_size` of .20 or an 80/20 train/test split ratio.  We add the `stratify` option set to `type_of_wine` to get similar proportions based on type.  

In [102]:
import numpy as np
from sklearn.model_selection import train_test_split

#Split the dataset into predictors (X) and response (y)
X_train, X_test, y_train, y_test = train_test_split(
    wine_df.drop(['alcohol'], axis=1),                 #prodictors(X)
    wine_df['alcohol'],                                #response
    test_size=0.2,                                     #80/20 split
    random_state=42,
    stratify=wine_df['type_of_wine']                   #maintains wine type proportions in each split
)

In [103]:
round(wine_df['type_of_wine'].value_counts()/len(wine_df['type_of_wine']), 3)

,count
type_of_wine,
1.0,0.754
0.0,0.246


In [104]:
round(X_train['type_of_wine'].value_counts()/len(X_train['type_of_wine']), 3)

,count
type_of_wine,
1.0,0.754
0.0,0.246


In [105]:
round(X_test['type_of_wine'].value_counts()/len(X_test['type_of_wine']), 3)

,count
type_of_wine,
1.0,0.754
0.0,0.246


The updated wine proportions match the expected values after stratified sampling, confirming this step.

### Standardize Predictors
Since the the variables in the wine_df dataset are on very different scales, we will standardize the predictors to imporve model performance and ensures that all features contribute eaually.
- We store the `means` and `stds` used for standardization, so we can standardize the test set using the same values.

In [106]:
means = X_train.apply(np.mean, axis = 0)
means

,0
fixed_acidity,7.215836
volatile_acidity,0.338770
citric_acid,0.318378
residual_sugar,5.392101
chlorides,0.056070
free_sulfur_dioxide,30.524533
total_sulfur_dioxide,115.750337
density,0.994678
pH,3.217976
sulphates,0.531155


In [107]:
stds = X_train.apply(np.std, axis = 0)
stds

,0
fixed_acidity,1.300954
volatile_acidity,0.164631
citric_acid,0.143717
residual_sugar,4.725284
chlorides,0.035602
free_sulfur_dioxide,17.771404
total_sulfur_dioxide,56.743248
density,0.003018
pH,0.160134
sulphates,0.147937


- Standardize the training predictors by subtracting each column's mean and dividing by its standard deviation.

In [108]:
X_train = X_train.apply(lambda x: (x-np.mean(x))/np.std(x), axis = 0)
X_train.head()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,quality,type_of_wine
4110,-0.780839,-0.235499,1.194170,1.229111,0.054220,2.108751,1.749101,0.623768,-0.174702,0.465371,0.201651,0.571351
3062,0.833361,-0.903661,1.194170,-0.887164,-0.142398,-1.098649,0.374488,-0.953600,-0.986520,-0.480979,0.201651,0.571351
5236,1.140827,-0.235499,0.150449,-0.125305,-1.069314,-0.423407,0.198256,0.020657,-0.611835,-0.278190,1.341998,0.571351
497,-0.012173,0.007469,0.011287,-0.612048,0.953048,0.701997,-0.048470,0.637023,0.637117,1.749702,-0.938696,-1.750237
826,0.218427,-0.417725,0.150449,-0.654374,-0.170487,-1.492540,-1.898910,0.139954,1.136697,0.735756,1.341998,-1.750237


In [109]:
#now each column has mean 0
X_train.apply(np.mean, axis = 0)

,0
fixed_acidity,1.011741e-16
volatile_acidity,-6.699364e-17
citric_acid,-1.066429e-16
residual_sugar,3.691486e-17
chlorides,5.742312e-17
free_sulfur_dioxide,9.023633e-17
total_sulfur_dioxide,6.562642e-17
density,-2.437065e-14
pH,2.604549e-16
sulphates,-1.517611e-16


In [110]:
#now each column has sd 1
X_train.apply(np.std, axis = 0)

,0
fixed_acidity,1.0
volatile_acidity,1.0
citric_acid,1.0
residual_sugar,1.0
chlorides,1.0
free_sulfur_dioxide,1.0
total_sulfur_dioxide,1.0
density,1.0
pH,1.0
sulphates,1.0


## Regression Task     
We fit the regression models using `alcohol` as the response variable.

### Train Models



#### Fit four different MLR Models
We will fit four different multiple linear regression (MLR) models. The first model will include all predictors, the second will include only the key predictors related to alcohol, the third will incorporate interaction terms, and the fourth will include selected polynomial terms. All predictors will be standardized, and cross-validation (CV) will be used to select the best-performing MLR model. This best model will then be compared with other top models on the test dataset to determine the overall winner.


For all four MLR models we use `cross_validate` function from `sklearn.model.selection` to do 5 fold CV with `neg_mean_squared_error` as the scoring criteria. Each model uses `LinearRegression()` from `sklearn.linear_model` with the same y response of `alchohol` from `y_train`.

First, we will fit a full model includsing all predictors.

In [111]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LinearRegression

# Full model
cv_full_model = cross_validate(
    LinearRegression(),
    X_train,
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")   #evaluation metric

Second, we include only the most important wine-content features, `density and residual_sugar`, as predictors.

In [112]:
#reduced model
cv_mlr1= cross_validate(
    LinearRegression(),
    X_train[['density','residual_sugar', 'fixed_acidity']],  #predictors
    y_train,
    cv = 5,
    scoring = "neg_mean_squared_error")                      #evaluation metric

We generated an interaction term between `density and residual_sugar` using `PolynomialFeatures` with interaction-only mode. The interaction was fit on the training set to avoid data leakage and then added back into the predictor matrix before running the multiple linear regression model.

We will include a interation term in the modle.
- To include interaction terms between `density and residual_sugar` , we use `PolynomialFeatures()` from `sklearn` tools to generate them.
 - First, we create interaction transformer

In [113]:
from sklearn.preprocessing import PolynomialFeatures
#create an interaction term transformer
interaction = PolynomialFeatures(interaction_only=True, include_bias = False)

- Generate transform X_train data

In [114]:
# Generate the interaction features for the training data (fit and transform)
inter_train = interaction.fit_transform(X_train[["density", "fixed_acidity", "residual_sugar"]])

- Check the feature names

In [115]:
# Create DataFrames with feature names
feature_names = interaction.get_feature_names_out(
    ['density', 'fixed_acidity', 'residual_sugar']
)
feature_names

array(['density', 'fixed_acidity', 'residual_sugar',
       'density fixed_acidity', 'density residual_sugar',
       'fixed_acidity residual_sugar'], dtype=object)

- keep the columns that we want for modeling

In [116]:
import pandas as pd
#Keep only the desired predictors
cols_to_keep = ['density', 'fixed_acidity', 'residual_sugar','fixed_acidity residual_sugar']

X_train_inter = pd.DataFrame(inter_train, columns=feature_names)[cols_to_keep]

- Finally, we perform 5-cross-validation with LinearRegression using using the training data with the added interaction terms (X_train_inter) to fit the model `cv_mlr2`.

In [117]:
cv_mlr2 = cross_validate(
    LinearRegression(),
    X_train_inter,
    y_train,
    cv=5,
    scoring="neg_mean_squared_error"    #evaluation metric
)

Now, we incorporate a polynomial term (density²) into the multiple linear regression model, using `density and residual_sugar` as predictors.

- We use `PolynomialFeatures()` to create polynomial terms, including the squared term for density.

In [118]:
poly = PolynomialFeatures(
    degree=2,
    interaction_only=False,  #allow squared terms
    include_bias=False
)

- Generate transform transing data

In [119]:
poly_train = poly.fit_transform(X_train[["density", "residual_sugar"]])

- Let us check the feature names

In [120]:
poly.get_feature_names_out(["density", "residual_sugar"])

array(['density', 'residual_sugar', 'density^2', 'density residual_sugar',
       'residual_sugar^2'], dtype=object)

- keet the columns (density, residual_sugar, desnsity^2) for modeling

In [121]:
import pandas as pd

cols_to_keep = ["density", "residual_sugar", "density^2"]

X_train_poly = pd.DataFrame(poly_train,
                            columns=poly.get_feature_names_out(["density","residual_sugar"])
                           )[cols_to_keep]

- Finally, we perform 5-cross-validation with LinearRegression using using the training data with the added polynomial terms (X_train_inter) to fit the model cv_mlr3.

In [122]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LinearRegression
#fit the model
cv_mlr3 = cross_validate(
    LinearRegression(),
    X_train_poly,
    y_train,
    cv=5,
    scoring="neg_mean_squared_error"     #evaluation metric
)

- We compute the cross‑validated RMSE for each of the four fitted multiple linear regression models to compare their predictive performance.

In [123]:
print(
    f"{np.sqrt(-sum(cv_full_model['test_score'])):.4f}",
    f"{np.sqrt(-sum(cv_mlr1['test_score'])):.4f}",
    f"{np.sqrt(-sum(cv_mlr2['test_score'])):.4f}",
    f"{np.sqrt(-sum(cv_mlr3['test_score'])):.4f}"
)

1.1549 1.7775 1.7784 2.1324


Among the four models, `cv_full_model` achieves the lowest RMSE (1.1549), indicating the best predictive performance. The model `cv_mlr3` has the highest RMSE (2.1324), making it the poorest performer.


After selecting the full model via cross-alidation, we refit it on the entire training set so that we can evaluate its performance on the test set.

In [124]:
mlr_best = LinearRegression().fit(X_train, y_train)
print(mlr_best.intercept_)
print(np.array(list(zip(X_train.columns, mlr_best.coef_))))

10.489583092809905
[['fixed_acidity' '0.6628226753046493']
 ['volatile_acidity' '0.13659555386556077']
 ['citric_acid' '0.07698578383284707']
 ['residual_sugar' '1.0680497388874999']
 ['chlorides' '-0.038223319373377904']
 ['free_sulfur_dioxide' '-0.05550544789657755']
 ['total_sulfur_dioxide' '-0.01624484064385051']
 ['density' '-1.9543140082320571']
 ['pH' '0.41337475834032245']
 ['sulphates' '0.15157121014222186']
 ['quality' '0.08870369438131037']
 ['type_of_wine' '-0.4732831467916546']]


The full multiple linear regression model indicates that `density` has the strongest association with alcohol content, showing the largest negative coefficient (-1.9541). Among the positive predictors, residual_sugar is the most influential (coefficient = 1.068), followed by `fixed_acidity` and `pH`. `Total_sulfur_dioxide` also exhibits a moderate negative relationship with alcohol content. In contrast, variables such as `chlorides` and `free_sulfur_dioxide` have very small coefficients, suggesting minimal impact on the response. The binary indicator type_of_wine also has a negative coefficient, implying that white wines (coded as 1) tend to have slightly lower alcohol levels than red wines (coded as 0), after controlling for the other predictors.

#### Fit a LASSO model
We use 5 predicors (`fixed_acidity, density, residual_sugar, total_sulfur_dioxide, and pH` ) to fit this model. Cross-validation (CV) will be applied to determine the optimal tuning parameter and identify the 'best' LASSO model. This selected model will then be compared with the best-performing models from other methods using the test set to determine the overall winner.





- Fit the LASSO model on those predictors using CV

In [125]:
from sklearn.linear_model import LassoCV, Lasso
#selected predictors
la_predictors = ['fixed_acidity','density', 'residual_sugar', 'total_sulfur_dioxide', 'pH']

lasso_mod = LassoCV(cv=5, random_state=0) \
    .fit(X_train[la_predictors],
         y_train)

Check the tuning parameters and CV errors

In [126]:
np.set_printoptions(suppress = True)
fit_info = np.array(list(zip(lasso_mod.alphas_, np.mean(lasso_mod.mse_path_, axis = 1))))
fit_info[fit_info[:,1].argsort()][0:10,].round(4)

array([[0.0008, 0.3768],
       [0.0009, 0.3768],
       [0.0009, 0.3768],
       [0.001 , 0.3768],
       [0.0011, 0.3768],
       [0.0012, 0.3768],
       [0.0013, 0.3768],
       [0.0013, 0.3768],
       [0.0014, 0.3769],
       [0.0015, 0.3769]])

We extracted the cross-validated MSE corresponding to each candidate value of the LASSO tuning parameter $\alpha$. The lowest cross-validated MSE (0.3768) occurs at $\alpha$ = 0.
0008, indicating that the LASSO penalty has a samll effect on the model.

In [127]:
np.set_printoptions(suppress = True)
print(lasso_mod.alpha_)
print(lasso_mod.intercept_)
print(np.array(list(zip(X_train[la_predictors].columns, np.round(lasso_mod.coef_, 4)))))

0.0008255721135897033
10.489583092809914
[['fixed_acidity' '0.7332']
 ['density' '-1.6203']
 ['residual_sugar' '0.8327']
 ['total_sulfur_dioxide' '-0.3127']
 ['pH' '0.4974']]


Among the five predictors included in the LASSO model, `density` shows the strongest negative association with alcohol content (coefficient = -1.6203). In contrast, `total_sulfur_dioxide` has the weakest negative effect (coefficient = -0.3127).

- Fit that best model for comparison on the test set uisng the optimal tuning parameter (α=0.0008) selected by CV.

In [128]:
lasso_best = Lasso(lasso_mod.alpha_) \
    .fit(X_train[la_predictors],
         y_train)

#### Fit a Ridge Regression model
We fit a Ridge regression model using five selected predictors (`fixed_acidity, density, quality, total_sulfur_dioxide, and pH`). The tuning parameter λ was chosen through 5-fold cross-validation via RidgeCV, using a log-spaced grid of candidate values from 10⁻³ to 10³. The optimal λ identified by CV is then used to fit the final model. This best-performing Ridge model will be compared with the top models obtained from other methods on the test set to determine the overall winner.


In [129]:
from sklearn.linear_model import Ridge
import numpy as np
import pandas as pd
from sklearn.linear_model import RidgeCV

# Set up a range of possible lambda values
alphas = np.logspace(-3, 3, 50)

# Initialize the RidgeCV model to find the best lambda
ridge_cv_model = RidgeCV(alphas=alphas, cv=5)

#selected predictors for ridge
ripredictors = ['fixed_acidity', 'density', 'quality', 'total_sulfur_dioxide', 'pH']

# Fit the model on the training data
ridge_cv_model.fit(X_train[ripredictors],
                   y_train)

# Print the best alpha (lambda) value
print(f"Optimal lambda: {ridge_cv_model.alpha_:.4f}")

Optimal lambda: 33.9322


Using 5-fold cross-validation, RidgeCV selected an optimal penalty parameter of λ = 33.9322. This value represents the amount of shrinkage that yields the lowest cross-validated mean squared error among the candidate λ values tested.

-  Fit the final Ridge regression model on the training data using the optimal tuning parameter (λ = 33.9322) selected by cross-validation.

In [130]:
#Fit final Ridge model using selected lambda
ridge_best = Ridge(ridge_cv_model.alpha_) \
            .fit(X_train[ripredictors],
                   y_train)

#### Fit an Elastic Net model
We use five selected predictors (`fixed_acidity, density, residual_sugar, chlorides, and sulphates`) to fit an Elastic Net model. Cross-validation (CV) is applied to determine the optimal tuning parameters, allowing us to identify the best Elastic Net model. This selected model will then be compared with the best-performing models from other methods on the test set to determine the overall winner.


- fit the model

In [131]:
from sklearn.linear_model import ElasticNetCV
#selected predixtors
enpredictors = ['fixed_acidity','density', 'residual_sugar', 'chlorides', 'sulphates']

regr = ElasticNetCV(cv=5,
                    random_state=0,
                    l1_ratio = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.96, 0.98, 0.99, 1],
                    n_alphas = 50).fit(X_train[enpredictors], y_train)

In [132]:
print(regr.alpha_)

0.003302288454358813


In [133]:
print(regr.l1_ratio_)

0.25


- Refit on full training data with best tuning parameters

In [134]:
from sklearn.linear_model import ElasticNet
en = ElasticNet(alpha = regr.alpha_, l1_ratio = regr.l1_ratio_)
en.fit(X_train[enpredictors], y_train)
print(en)

ElasticNet(alpha=np.float64(0.003302288454358813), l1_ratio=np.float64(0.25))


In [135]:
print(np.array(list(zip(X_train[enpredictors].columns, en.coef_))))

[['fixed_acidity' '0.4332072989443492']
 ['density' '-1.2871866456100858']
 ['residual_sugar' '0.37490842658466605']
 ['chlorides' '-0.03902128268781735']
 ['sulphates' '0.28344762105197835']]


### Test Models
To test our models properly, we standardize the X_test data using the means and standard deviations computed from the training set. This step is essential because the test data must be transformed in exactly the same way as the training data to ensure consistent and valid model evaluation.

#### Standardize the X_test data

In [136]:
#function to standardize based off of a supplied mean and std
def my_std_fun(x, means, stds):
    return(x-means)/stds
#loop through the columns and use the function on each
for x in X_test.columns:
    X_test[x] = my_std_fun(X_test[x], means[x], stds[x])

X_test.head()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,quality,type_of_wine
1739,-0.703973,-0.174757,0.150449,-0.675536,-0.310928,-0.592217,-0.682907,-0.655358,0.512221,-0.683768,-0.938696,0.571351
2845,0.602761,-0.842919,0.567938,-0.040654,-0.030045,0.589456,0.903185,0.206230,-0.237149,-0.886557,1.341998,0.571351
5282,-0.319639,-1.146628,0.637519,1.588878,-0.339017,2.755858,1.185157,0.908755,0.137536,1.682105,0.201651,0.571351
5822,-1.472639,0.554146,-1.449922,-0.908327,-0.760342,-1.380000,-1.141111,-1.298235,1.823620,-0.345786,-2.079043,0.571351
5237,-0.627106,-0.296241,0.011287,-0.633211,-1.181667,0.195565,-0.471428,-1.523573,0.137536,0.870949,1.341998,0.571351


Next, we generate predictions and compute the RMSE on the test set. For the lasso_best, ridge_best, and en models, we use only the corresponding subset of X_test, since these models were trained on a restricted set of features.

In [137]:
from sklearn.metrics import mean_squared_error
mlr_pred = mlr_best.predict(X_test)
lasso_pred = lasso_best.predict(X_test[la_predictors])
ridge_pred = ridge_best.predict(X_test[ripredictors])
en_pred= en.predict(X_test[enpredictors])
#Used the full X_test for mlr_best, which was trained on all features.
print(np.sqrt(mean_squared_error(y_test, mlr_pred)),
      np.sqrt(mean_squared_error(y_test, lasso_pred)),
      np.sqrt(mean_squared_error(y_test, ridge_pred)),
      np.sqrt(mean_squared_error(y_test, en_pred)))

0.4653460972974002 0.5886755570804444 0.7137168895585853 0.7549037847242995


Based on the RMSE values, the best-performing model is `mlr_best`, with the lowest RMSE of 0.4653, indicating the best predictive performance. The lasso_best performs moderately well with an RMSE of 0.5887. The ridge_best shows poorer performnace with an RMSE of 0.7172. Elastic net models has the highest RMSE (0.7549), making it the worst-performing model among those compared.

- Calculate MAE (Mean Absolute Error)

In [138]:
from sklearn.metrics import mean_absolute_error
mae_mlr = mean_absolute_error(y_test, mlr_pred)
mae_lasso = mean_absolute_error(y_test, lasso_pred)
mae_ridge = mean_absolute_error(y_test, ridge_pred)
mae_en = mean_absolute_error(y_test, en_pred)
print(mae_mlr, mae_lasso, mae_ridge, mae_en)

0.35078764817362956 0.44599288451961894 0.550958792860084 0.5878681494864484


The MLR model demonstrated the strongest performance, achiveing the lowest MAE of 0.3508. The Lasso model produced a higher MAE of 0.4460, followed by Ridge with an MAE of 0.5510, and the Elastic Net model with the highest MAE of 0.5879. These results indicate that, for this particular dataset and set of predictors, regularization did not improve predictive accuracy, and the full multiple linear regression model provided the most accurate predictions of alcohol content.

## Classification Task   
In this section, we will repeat the previous training and testing procedure, but now using logistic regression models, with `type_of_wine` as the response variable.

### Split the data

In [139]:
import numpy as np
from sklearn.model_selection import train_test_split

#Split the dataset into predictors (X) and response (y)
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    wine_df.drop(['type_of_wine'], axis=1),                #prodictors(X)
    wine_df['type_of_wine'],                               #response
    test_size=0.2,                                         #80/20 split
    random_state=42,
    stratify=wine_df['type_of_wine']                       #maintains wine type proportions in each split
)

### Train Models
Since the the variables in the wine_df dataset are on very different scales, we will standardize the predictors to imporve model performance and ensures that all features contribute eaually.
- We store the `means_clf` and `stds_clf` used for standardization, so we can standardize the test set using the same values.


In [140]:
means_clf = X_train_clf.apply(np.mean, axis = 0)
means_clf

,0
fixed_acidity,7.215836
volatile_acidity,0.338770
citric_acid,0.318378
residual_sugar,5.392101
chlorides,0.056070
free_sulfur_dioxide,30.524533
total_sulfur_dioxide,115.750337
density,0.994678
pH,3.217976
sulphates,0.531155


In [141]:
stds_clf = X_train_clf.apply(np.std, axis = 0)
stds_clf

,0
fixed_acidity,1.300954
volatile_acidity,0.164631
citric_acid,0.143717
residual_sugar,4.725284
chlorides,0.035602
free_sulfur_dioxide,17.771404
total_sulfur_dioxide,56.743248
density,0.003018
pH,0.160134
sulphates,0.147937


- Standardize the training predictors by subtracting each column's mean (means_clf) and dividing by its standard deviation (stds_clf).

In [142]:
X_train_clf = X_train_clf.apply(lambda x: (x-np.mean(x))/np.std(x), axis = 0)
X_train_clf.head()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
4110,-0.780839,-0.235499,1.194170,1.229111,0.054220,2.108751,1.749101,0.623768,-0.174702,0.465371,-0.912111,0.201651
3062,0.833361,-0.903661,1.194170,-0.887164,-0.142398,-1.098649,0.374488,-0.953600,-0.986520,-0.480979,0.427280,0.201651
5236,1.140827,-0.235499,0.150449,-0.125305,-1.069314,-0.423407,0.198256,0.020657,-0.611835,-0.278190,0.594703,1.341998
497,-0.012173,0.007469,0.011287,-0.612048,0.953048,0.701997,-0.048470,0.637023,0.637117,1.749702,0.510992,-0.938696
826,0.218427,-0.417725,0.150449,-0.654374,-0.170487,-1.492540,-1.898910,0.139954,1.136697,0.735756,0.427280,1.341998


#### Fit 4 logistic regression Models
We will fit four logistic regression models. The first model will include all available predictors. The second model will use only the key predictors most closely associated with the binary response variable type_of_wine (0 = red, 1 = white). The third model will add selected interaction terms, and the fourth will incorporate polynomial transformations of key predictors. All predictors will be standardized prior to model fitting. Cross-validation will be used to tune the logistic regression models and identify the best-performing one. Finally, the selected best model will be compared with the other candidate models using the test dataset to assess overall classification performance.

- The model `log_reg_cv1` is fit using all available predictors in the training dataset.

In [143]:
from sklearn.model_selection import cross_validate
from sklearn.linear_model import LogisticRegressionCV
# fit a full model
log_reg_cv1 = LogisticRegressionCV(
    cv=5,
    solver="newton-cholesky",
    penalty="l2",
    Cs=250,
    scoring="neg_log_loss",
    random_state=10,
    max_iter=5000
).fit(X_train_clf, y_train_clf)

- Use only key vaibales `density, residual_sugar, total_sulfur_dioxide, free_sulfur_dioxide, chlorides` to fit the log_reg_cv2 model

In [144]:
#fit model on selected predictors
log_reg_cv2 = LogisticRegressionCV(
    cv=5,
    solver="newton-cholesky",
    penalty="l2",
    Cs=250,
    scoring="neg_log_loss",
    random_state=10,
    max_iter=5000
).fit(X_train_clf[['density', 'residual_sugar', 'total_sulfur_dioxide', 'free_sulfur_dioxide', 'chlorides']],
                y_train_clf)

- Add interaction terms into logistic regression

- Initialize PolynomialFeatures to generate only interaction terms

In [145]:
from sklearn.preprocessing import PolynomialFeatures
interaction = PolynomialFeatures(interaction_only=True, include_bias=False)

- Fit the interaction transformer *only on the standardized training predictors*

In [146]:
interaction.fit(X_train_clf[['density','residual_sugar','total_sulfur_dioxide']])

PolynomialFeatures(include_bias=False, interaction_only=True)

In [147]:
#Check which interaction terms were generated
interaction.get_feature_names_out(['density','residual_sugar','total_sulfur_dioxide'])

array(['density', 'residual_sugar', 'total_sulfur_dioxide',
       'density residual_sugar', 'density total_sulfur_dioxide',
       'residual_sugar total_sulfur_dioxide'], dtype=object)

In [148]:
log_reg_cv3 = LogisticRegressionCV(
    cv=5,
    solver="newton-cholesky",
    penalty="l2",
    Cs=250,
    scoring="neg_log_loss",
    random_state=10,
    max_iter=5000
).fit(interaction.transform(X_train_clf[['density','residual_sugar','total_sulfur_dioxide']]),
                y_train_clf)

- Add polynomial terms into ligistic regression

In [149]:
#Initialize PolynomialFeatures to generate polynomial terms
poly = PolynomialFeatures(degree=2, interaction_only=False, include_bias=False)

In [150]:
#Fit the transformer on the training data
poly.fit(X_train_clf[['density','residual_sugar','total_sulfur_dioxide']])

PolynomialFeatures(include_bias=False)

In [151]:
#Check which interaction terms were generated
poly.get_feature_names_out(['density','residual_sugar','total_sulfur_dioxide'])

array(['density', 'residual_sugar', 'total_sulfur_dioxide', 'density^2',
       'density residual_sugar', 'density total_sulfur_dioxide',
       'residual_sugar^2', 'residual_sugar total_sulfur_dioxide',
       'total_sulfur_dioxide^2'], dtype=object)

- keet the columns (density, residual_sugar, total_sulfur_dioxide, residual_sugar^2, desnsity^2) for modeling

In [152]:
import pandas as pd

cols_to_keep = ['density', 'residual_sugar', 'total_sulfur_dioxide', 'residual_sugar^2', 'density^2'] # Corrected typo 'desnsity^2' to 'density^2'

#transform X_train using the polynomial transformer
poly_features_data = poly.transform(X_train_clf[['density','residual_sugar','total_sulfur_dioxide']])

#extract feature names generated by PolynomialFeatures
poly_feature_names = poly.get_feature_names_out(['density','residual_sugar','total_sulfur_dioxide'])

# Create a DataFrame from the polynomial features
X_train_poly = pd.DataFrame(poly_features_data, columns=poly_feature_names)

# Keep only the selected columns
X_train_poly = X_train_poly[cols_to_keep]
X_train_poly.head()

,density,residual_sugar,total_sulfur_dioxide,residual_sugar^2,density^2
0,0.623768,1.229111,1.749101,1.510714,0.389087
1,-0.953600,-0.887164,0.374488,0.787060,0.909352
2,0.020657,-0.125305,0.198256,0.015701,0.000427
3,0.637023,-0.612048,-0.048470,0.374603,0.405799
4,0.139954,-0.654374,-1.898910,0.428205,0.019587


We can see that the polynomial features (squared and interaction terms) have been successfully added.

In [153]:
log_reg_cv4 = LogisticRegressionCV(
    cv=5,
    solver="newton-cholesky",
    penalty="l2",
    Cs=250,
    scoring="neg_log_loss",
    random_state=10,
    max_iter=5000
).fit(X_train_poly, y_train_clf)

- we will write a help function to extract the best CV score

In [154]:
def best_cv_score(logreg_model, positive_class=1):
    #mean neg_log_loss for each C
    means = logreg_model.scores_[positive_class].mean(axis=0)
    #best mean score
    return means.max()

- pick the best model

In [155]:
print(
    f"{best_cv_score(log_reg_cv1):.4f}",
    f"{best_cv_score(log_reg_cv2):.4f}",
    f"{best_cv_score(log_reg_cv3):.4f}",
    f"{best_cv_score(log_reg_cv4):.4f}"
)

-0.0441 -0.0688 -0.0686 -0.0684


Therefore, the log_reg_cv1 model, which uses all available predictors, achieved the best performance with a negative log-loss of -0.0441. The models incorporating interaction or polynomial terms, as well as the one with selected predictors, performed similarly to each other but slightly worse than the full model.

Among the four models, log_reg_cv1, which uses all available predictors, achieved the best cross-validated performance with a negative log-loss of -0.0441.   

The models using selected predictors (log_reg_cv2), interaction-only terms (log_reg_cv3), and degree-2 polynomial terms (log_reg_cv4) all performed similarly to one another, but each produced noticeably worse log-loss values compared to the full model.

This indicates that reducing predictors or adding interactions/polynomials did not improve predictive performance, and the simplest and most inclusive approach (log_reg_cv1) generalizes best in this case.

- Refit the best-performing model (cv1) on the full X_train_clf dataset to obtain the final model that will be used for evaluation on the test set.

In [156]:
from sklearn.linear_model import LogisticRegression
#Refit a final logistic regression model using the optimal C from CV
best_log_reg = LogisticRegression(
    penalty='l2',
    solver="newton-cholesky",
    C=log_reg_cv1.C_[0],
    max_iter=5000
).fit(X_train_clf, y_train_clf)

#### Fit a LASSO logistic regression model
In this section, we use five predictors, `residual_sugar, free_sulfur_dioxide, fixed_acidity, density, and volatile_acidity` to fit a LASSO-regularized logistic regression model, where the binary variable `type_of_wine` serves as the response. Cross-validation is used to select the optimal tuning parameter (λ), allowing us to identify the best LASSO model. The selected model will then be evaluated against the best-performing models from the other methods using the test set to determine the overall winner.


- Fit a LASSO-penalized logistic regression model using cross-validation on the selected predictors.

In [157]:
from sklearn.linear_model import LogisticRegressionCV

lasso_predictors = ['residual_sugar', 'free_sulfur_dioxide',
                    'fixed_acidity', 'density', 'volatile_acidity']
#fit model
lasso_logit = LogisticRegressionCV(
    penalty='l1',
    solver='liblinear',
    cv=5,
    scoring='neg_log_loss',
    max_iter=5000
    ).fit(X_train_clf[lasso_predictors], y_train_clf)

print(lasso_logit.C_)       # best C
print(lasso_logit.coef_)    # coefficients

[0.35938137]
[[ 3.71424894  0.96500221 -0.12494692 -3.55693554 -1.43446957]]


- Refit the LASSO logistic regression model on the selected predictors using the optimal tuning parameter selected by CV

In [158]:
lasso_logit_best = LogisticRegression(
    penalty='l1',
    solver='liblinear',
    C=lasso_logit.C_[0],  #optimal C
    max_iter=5000
).fit(X_train_clf[lasso_predictors], y_train_clf)

#### Fit a Ridge Logistic Regression Model  
We will use six predictors, `density, pH, residual_sugar, total_sulfur_dioxide, citric_acid, and sulphates` to fit a Ridge logistic regression model. Using the optimal tuning parameter selected by cross-validation, we will refit the best model for evaluation on the test set.

In [159]:
from sklearn.linear_model import LogisticRegressionCV
#subset the training data to only include selected predictors
ridge_predictors =['density', 'pH', 'residual_sugar',
                  'total_sulfur_dioxide', 'citric_acid', 'sulphates']

ridge_log = LogisticRegressionCV(
    penalty='l2',         #Ridge penalty
    solver='lbfgs',
    cv=5,
    scoring='neg_log_loss',
    Cs=50,                # number of C values to search
    max_iter=5000,
    random_state=42
).fit(X_train_clf[ridge_predictors], y_train_clf)

- Refit the Ridge logistic regression model on the selected predictors using the optimal tuning parameter selected by CV

In [160]:
ridge_log_best = LogisticRegression(
    penalty='l2',
    solver='lbfgs',
    C=ridge_log.C_[0],              #optimal C
    max_iter=5000
).fit(X_train_clf[ridge_predictors], y_train_clf)

#### Fit an Elastic Net Logistic Regression model   
We will use five predictors, `density, residual_sugar, total_sulfur_dioxide, free_sulfur_dioxide', and 'volatile_acidity`, to fit an Elastic Net logistic regression  model. Using the optimal tuning parameter selected by cross-validation, we will refit the best model for evaluation on the test set.


In [161]:
# Elastic Net Logistic Regression
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

#subset the training data to only include selected predictors
logen_predictors = ['density', 'residual_sugar','total_sulfur_dioxide',
                          'free_sulfur_dioxide', 'volatile_acidity']

# Define the Logistic Regression model for Elastic Net
# Note: l1_ratio and C will be tuned by GridSearchCV
log_reg_en = LogisticRegression(
    penalty='elasticnet',
    solver='saga',
    random_state=42,
    max_iter=5000 # Increased max_iter for convergence
)

# Define the parameter grid to search over
param_grid = {
    'l1_ratio': [0.1, 0.5, 0.9, 0.95, 0.99, 1.0], # l1_ratio values to test
    'C': np.logspace(-4, 4, 10)                   # C values to test
}

#Fit an Elastic Net logistic regression model using 5-fold CV to tune C and l1_ratio based on negative log-loss
en_log = GridSearchCV(
    log_reg_en,
    param_grid,
    cv=5,
    scoring='neg_log_loss',
    n_jobs=-1                # Use all available cores
).fit(X_train_clf[logen_predictors], y_train_clf)

In [162]:
#Retrieve the best model automatically refit by GridSearchCV.
en_log_best = en_log.best_estimator_

We don't need to refit the Elastic Net model manually, as `GridSearchCV()` automatically refits the estimator on the full training set using the best hyperparameters (refit=True by default).

### Test Models
To test our models properly, we standardize the X_test_clf data using the means and standard deviations computed from the training set. This step is essential because the test data must be transformed in exactly the same way as the training data to ensure consistent and valid model evaluation.

- We will use previouse defined function 'my_std_fun' to standardize the X_test dataset.

In [163]:
for x in X_test_clf.columns:
    X_test_clf[x] = my_std_fun(X_test_clf[x], means_clf[x], stds_clf[x])

X_test_clf.head()

,fixed_acidity,volatile_acidity,citric_acid,residual_sugar,chlorides,free_sulfur_dioxide,total_sulfur_dioxide,density,pH,sulphates,alcohol,quality
1739,-0.703973,-0.174757,0.150449,-0.675536,-0.310928,-0.592217,-0.682907,-0.655358,0.512221,-0.683768,-0.242415,-0.938696
2845,0.602761,-0.842919,0.567938,-0.040654,-0.030045,0.589456,0.903185,0.206230,-0.237149,-0.886557,0.092432,1.341998
5282,-0.319639,-1.146628,0.637519,1.588878,-0.339017,2.755858,1.185157,0.908755,0.137536,1.682105,-0.242415,0.201651
5822,-1.472639,0.554146,-1.449922,-0.908327,-0.760342,-1.380000,-1.141111,-1.298235,1.823620,-0.345786,0.594703,-2.079043
5237,-0.627106,-0.296241,0.011287,-0.633211,-1.181667,0.195565,-0.471428,-1.523573,0.137536,0.870949,1.682958,1.341998


- Assess each model on the test set using two metrics: log-loss and accuracy.


In [164]:
# logistic regression (full predictors)
log_reg_pred = best_log_reg.predict_proba(X_test_clf)

# lasso predictors subset
X_test_lasso = X_test_clf[lasso_predictors]
lasso_log_pred = lasso_logit_best.predict_proba(X_test_lasso)

# ridge predictors subset
X_test_ridge = X_test_clf[ridge_predictors]
ridge_log_pred = ridge_log_best.predict_proba(X_test_ridge)

# elastic net predictors subset
X_test_logen = X_test_clf[logen_predictors]
en_log_pred = en_log_best.predict_proba(X_test_logen)

In [165]:
from sklearn.metrics import log_loss, accuracy_score
print("LOG-LOSS:")
print(
    f"{log_loss(y_test_clf, log_reg_pred):.4f}",
    f"{log_loss(y_test_clf, lasso_log_pred):.4f}",
    f"{log_loss(y_test_clf, ridge_log_pred):.4f}",
    f"{log_loss(y_test_clf, en_log_pred):.4f}"
)

print("\nACCURACY:")
print(
    f"{accuracy_score(y_test_clf, best_log_reg.predict(X_test_clf)):.4f}",
    f"{accuracy_score(y_test_clf, lasso_logit_best.predict(X_test_lasso)):.4f}",
    f"{accuracy_score(y_test_clf, ridge_log_best.predict(X_test_ridge)):.4f}",
    f"{accuracy_score(y_test_clf, en_log_best.predict(X_test_logen)):.4f}"
)

LOG-LOSS:
0.0187 0.0927 0.0445 0.0365

ACCURACY:
0.9962 0.9669 0.9838 0.9877


Across all models, the full logistic regression achieved the best performance with the lowest test log-loss (0.0187) and highest accuracy (0.9962), while Elastic Net and Ridge also performed strongly; LASSO performed worst.

# **Homework 8**

## Instruction
The purpose of this homework is to practice fitting tree based models using `csikit-learn`.
We will use the ` Wine Quality` dataset from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/186/wine+quality) that we used in hommwork 7.

we will use `alcohol` as a target variable for fitting regression type models while using `type of wine` as the response variable for fitting classification type models.
We will then compare all models performance including the models from homework 7 to find the best winner!

## Regression Task (alcohol as Response)

### Train Models

#### Fit a regression tree
We use five predictors (`fixed_acidity, density, residual_sugar, total_sulfur_dioxide, pH`) to fit a regression tree. We use `GridSearchCV` to select the tuning parameters (max_depth and min_samples_leaf). The 5-fold cross-validation is used. the `neg_mean_squared_error` is used as the score metric.

In [166]:
from sklearn.model_selection import GridSearchCV, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

#selected 5 predictors for the regression tree
rtree_predictors = ['fixed_acidity','density', 'residual_sugar', 'total_sulfur_dioxide', 'pH']

#Use GridSearchCV to tune the regression tree hyperparameters
rtree_tune = GridSearchCV(DecisionTreeRegressor(),
                            {'max_depth': range(2,15),'min_samples_leaf':[3, 10, 50, 100]}, cv = 5,
                            scoring = "neg_mean_squared_error") \
                            .fit(X_train[rtree_predictors],
                                 y_train)

- Display the best-performing tree and its tuned parameters


In [167]:
print(rtree_tune .best_estimator_)

DecisionTreeRegressor(max_depth=14, min_samples_leaf=10)


In [181]:
#Perform cross-validation using the best regression tree found
rtree_cv = cross_validate(rtree_tune.best_estimator_,
                          X_train[rtree_predictors],
                          y_train,
                          cv=5,
                          scoring='neg_mean_squared_error')

In [186]:
#check RMSA
print(
    f"{np.sqrt(-sum(rtree_cv['test_score'])/5):.4f}")

0.5521


In [185]:
#from sklearn.tree import plot_tree
# plot_tree(rtree_tune.best_estimator_)

#### Fit a random forest model

We use 6 predictors (`volatile_acidity, residual_sugar, density, total_sulfur_dioxide, sulphates`) to fit a random forest model. `GridSearchCV`() is used to select the best random fores modle

In [189]:
from sklearn.ensemble import RandomForestRegressor
#set up parameters
parameters = {'max_features': range(1,4),
              'max_depth': [3, 4, 5, 10, 15],
              'min_samples_leaf':[3, 10, 50, 100]}

#selected 5 presictors for randon forest model
rf_predictors = ['volatile_acidity', 'residual_sugar', 'density', 'total_sulfur_dioxide', 'sulphates' ]

#Initialize GridSearchCV
rf_tune = GridSearchCV(RandomForestRegressor(n_estimators = 500),
                       parameters, cv = 5, scoring = "neg_mean_squared_error")

In [190]:
#Fit the model
rf_tune.fit(X_train[rf_predictors],
            y_train)

GridSearchCV(cv=5, estimator=RandomForestRegressor(n_estimators=500),
             param_grid={'max_depth': [3, 4, 5, 10, 15],
                         'max_features': range(1, 4),
                         'min_samples_leaf': [3, 10, 50, 100]},
             scoring='neg_mean_squared_error')

In [191]:
print("Best Parameters:", rf_tune.best_estimator_)

Best Parameters: RandomForestRegressor(max_depth=15, max_features=3, min_samples_leaf=3,
                      n_estimators=500)


In [192]:
rf_cv = cross_validate(rf_tune.best_estimator_,
                        X_train[rf_predictors],
                        y_train, cv = 5,
                        scoring = "neg_mean_squared_error")

In [193]:
print(
    f"{np.sqrt(-sum(rf_cv['test_score'])/5):.4f}")

0.4934


### Test Models
We will use standardized X_test dataset to to test our models properly , then compare their performacen with another 4 models from homework7

In [195]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
rtree_preds = rtree_tune.best_estimator_.predict(X_test[rtree_predictors])
rf_preds = rf_tune.best_estimator_.predict(X_test[rf_predictors])

In [196]:
#Used the full X_test for mlr_best, which was trained on all features.
print(np.sqrt(mean_squared_error(y_test, mlr_pred)),
      np.sqrt(mean_squared_error(y_test, lasso_pred)),
      np.sqrt(mean_squared_error(y_test, ridge_pred)),
      np.sqrt(mean_squared_error(y_test, en_pred)),
      np.sqrt(mean_squared_error(y_test, rtree_preds)),
      np.sqrt(mean_squared_error(y_test, rf_preds)))

0.4653460972974002 0.5886755570804444 0.7137168895585853 0.7549037847242995 0.5454319979104493 0.4986846920929189


In [197]:
from sklearn.metrics import log_loss, accuracy_score
print("RSME:")
print(
    f"{np.sqrt(mean_squared_error(y_test, mlr_pred)):.4f}",
    f"{np.sqrt(mean_squared_error(y_test, lasso_pred)):.4f}",
    f"{np.sqrt(mean_squared_error(y_test, ridge_pred)):.4f}",
    f"{np.sqrt(mean_squared_error(y_test, en_pred)):.4f}",
    f"{np.sqrt(mean_squared_error(y_test, rtree_preds)):.4f}",
    f"{np.sqrt(mean_squared_error(y_test, rf_preds)):.4f}"
)

print("\nMAE:")
print(
    f"{mean_absolute_error(y_test, mlr_pred):.4f}",
    f"{mean_absolute_error(y_test, lasso_pred):.4f}",
    f"{mean_absolute_error(y_test, ridge_pred):.4f}",
    f"{mean_absolute_error(y_test, en_pred):.4f}",
    f"{mean_absolute_error(y_test, rtree_preds):.4f}",
    f"{mean_absolute_error(y_test, rf_preds):.4f}"
)

RSME:
0.4653 0.5887 0.7137 0.7549 0.5454 0.4987

MAE:
0.3508 0.4460 0.5510 0.5879 0.3945 0.3563
